In [10]:
from google.cloud import bigquery

In [11]:
# Nome do Projeto dentro do BQ
client = bigquery.Client(project="portfolio-marketing-digital")

#Pega a tabela que iremos trabalhar
table = client.get_table(
    "bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210131"
)

In [12]:
#abrindo a tabela | Field = Coluna | Field Type = Tipo de dado | Mode

for field in table.schema:
    print(field.name, field.field_type, field.mode)

    if field.field_type == "RECORD":          # ← 4 espaços: dentro do for
        for subfield in field.fields:         # ← 8 espaços
            print("    ", subfield.name, subfield.field_type, subfield.mode)  # ← 12


event_date STRING NULLABLE
event_timestamp INTEGER NULLABLE
event_name STRING NULLABLE
event_params RECORD REPEATED
     key STRING NULLABLE
     value RECORD NULLABLE
event_previous_timestamp INTEGER NULLABLE
event_value_in_usd FLOAT NULLABLE
event_bundle_sequence_id INTEGER NULLABLE
event_server_timestamp_offset INTEGER NULLABLE
user_id STRING NULLABLE
user_pseudo_id STRING NULLABLE
privacy_info RECORD NULLABLE
     analytics_storage INTEGER NULLABLE
     ads_storage INTEGER NULLABLE
     uses_transient_token STRING NULLABLE
user_properties RECORD REPEATED
     key INTEGER NULLABLE
     value RECORD NULLABLE
user_first_touch_timestamp INTEGER NULLABLE
user_ltv RECORD NULLABLE
     revenue FLOAT NULLABLE
     currency STRING NULLABLE
device RECORD NULLABLE
     category STRING NULLABLE
     mobile_brand_name STRING NULLABLE
     mobile_model_name STRING NULLABLE
     mobile_marketing_name STRING NULLABLE
     mobile_os_hardware_model INTEGER NULLABLE
     operating_system STRING NULLA

In [13]:
# Lê 5 linhas de algumas colunas, sem query (grátis)
rows = client.list_rows(
    table, 
    selected_fields=[table.schema[i] for i in (0, 2, 9, 14, 15)],
    max_results=5)  # ← 5 linhas
rows.to_dataframe()

,event_date,event_name,user_pseudo_id,device,geo
0,20210131,page_view,1026454.4271112504,"{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No..."
1,20210131,scroll,1026454.4271112504,"{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No..."
2,20210131,page_view,1026454.4271112504,"{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No..."
3,20210131,user_engagement,1026454.4271112504,"{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No..."
4,20210131,session_start,1026454.4271112504,"{'category': 'mobile', 'mobile_brand_name': 'A...","{'continent': 'Americas', 'sub_continent': 'No..."


In [14]:
query = """
select event_name, count(*) as events
from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
where _TABLE_SUFFIX between '20201101' and '20210131'
group by event_name
order by events desc
"""

job_config = bigquery.QueryJobConfig(dry_run=True,use_query_cache=False)

job = client.query(query, job_config=job_config)

print(f"Vai ler {job.total_bytes_processed / 1024**2:.1f} MiB")

Vai ler 53.4 MiB


In [15]:
df_events = client.query(query).to_dataframe()
df_events


c:\Users\dante\Projetos\marketing-digital\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,event_name,events
0,page_view,1350428
1,user_engagement,1058721
2,scroll,493072
3,view_item,386068
4,session_start,354970
5,first_visit,257462
6,view_promotion,190104
7,add_to_cart,58543
8,begin_checkout,38757
9,select_item,31007


In [ ]:
#Tabelas do Dataset

tables = list(client.list_tables("bigquery-public-data.ga4_obfuscated_sample_ecommerce"))

print(len(tables))
print(tables[0].table_id)
print(tables[-1].table_id)



92
events_20201101
events_20210131


In [25]:
query = """

select
    ep.key,
    count(*) as occurrences,
    countif(ep.value.string_value is not null) as as_string,
    countif(ep.value.int_value is not null) as as_int,
    countif(ep.value.double_value is not null) as as_double
from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`,
unnest(event_params) as ep
where _TABLE_SUFFIX between '20201101' and '20210131'
group by ep.key
order by occurrences desc
"""
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
job = client.query(query, job_config=job_config)

print(f"vai ler {job.total_bytes_processed / 1024**2:.1f} MiB")

vai ler 1345.8 MiB
